# Audit a Sage Brandt matrix and check an LMFDB coefficient

Install the project wheel into your Jupyter kernel's Python environment and open this notebook alongside `brandt_reference.json`. The saved replay works offline with ordinary Python. Select a SageMath kernel with the wheel installed to regenerate the Brandt data in the last cell.

For a weight-two operator at a good prime $\ell$, row $i$ counts neighbors from class $i$ to its targets. Every row should sum to $\ell+1$. For full stabilizer orders $d_i$, the kernel checks $d_j B_{ij}=d_i B_{ji}$ and $\sum_i 1/d_i$. The expected mass and correctness of the geometric tables are inputs to the audit.

Sage's `monodromy_weights()` gives orders modulo the central sign in these examples. Double them for the full convention used here. Use `hecke_matrix(ell)` directly for the row convention. [Sage Brandt modules](https://doc.sagemath.org/html/en/reference/modfrm/sage/modular/quatalg/brandt.html)

In [ ]:
import json
from fractions import Fraction
from pathlib import Path

from hecke_mass.adapters.brandt import audit_brandt_matrix

records = json.loads(Path("brandt_reference.json").read_text())


def audit_record(record):
    return audit_brandt_matrix(
        record["stabilizer_orders"],
        record["matrix"],
        prime=record["prime"],
        expected_mass=Fraction(record["expected_mass"]),
    )


for name, record in records.items():
    result = audit_record(record)
    assert result.ok, result
    print(name, "mass", result.mass, "row sums", tuple(map(str, result.row_sums)))

The level-23 orders, mass, and prime-2 matrix come from the Lean replay. Transposing that matrix preserves its characteristic polynomial but changes the row sums from $(3,3,3)$ to $(2,6,1)$ and breaks detailed balance against the recorded orders. [Lean replay](../../lean/HeckeMassKernel/ReplayedCases.lean)

In [ ]:
level23 = records["level_23"]
transposed = audit_record({**level23, "matrix": zip(*level23["matrix"])})
assert transposed.row_sums == (2, 6, 1)
assert not transposed.detailed_balance and not transposed.ok
print("Transposed row sums", tuple(map(str, transposed.row_sums)))

For level 11, Gross gives the matrix and the two stabilizer orders modulo sign. The common row sum 3 is one eigenvalue. The trace is the sum of both eigenvalues, so the remaining one can be compared exactly to $a_2=-2$ in the published expansion of `11.2.a.a`. The determinant provides a second check. [Gross, §6](https://wstein.org/papers/bib/Gross-Heights_and_the_Special_Values_of_L-series.pdf) · [LMFDB newform 11.2.a.a](https://www.lmfdb.org/ModularForm/GL2/Q/holomorphic/11/2/a/a/)

In [ ]:
level11 = records["level_11"]
(a, b), (c, d) = level11["matrix"]
degree = level11["prime"] + 1
coefficient = a + d - degree
assert a * d - b * c == degree * coefficient
assert coefficient == level11["newform_coefficient"]
print(level11["newform_label"], "a_2 =", coefficient)

The following exporter uses Sage's integer ring `ZZ` to check exact integrality before converting entries to Python integers. It audits the regenerated data and compares it to the saved fixture. The full orders are distinct within each of these two examples, so they identify the simultaneous row and column permutation if Sage enumerates its basis differently.

In [ ]:
try:
    from sage.all import BrandtModule, ZZ
except ModuleNotFoundError as error:
    if error.name not in ("sage", "sage.all"):
        raise
    print(
        "Select a SageMath kernel to run the live export. The saved replay is complete."
    )
else:
    for level in (11, 23):
        reference = records[f"level_{level}"]
        module = BrandtModule(level)
        orders = tuple(2 * int(w) for w in module.monodromy_weights())
        rows = [
            tuple(int(ZZ(entry)) for entry in row)
            for row in module.hecke_matrix(reference["prime"]).rows()
        ]
        regenerated = {**reference, "stabilizer_orders": orders, "matrix": rows}
        assert audit_record(regenerated).ok
        permutation = [orders.index(order) for order in reference["stabilizer_orders"]]
        assert [[rows[i][j] for j in permutation] for i in permutation] == reference[
            "matrix"
        ]
        print(f"Sage level {level} matches the saved reference")